# Notebook 3 · Seleção de variáveis

Objetivo: perceber se podemos reduzir as variáveis preparadas no NB_2 sem perder capacidade de prever uma nova compra nos 90 dias seguintes. Comparamos três abordagens estudadas nas aulas: **filter** (teste F), **wrapper** (RFE) e **embedded** (regressão logística com penalização L1).

Usamos a regressão logística apenas como modelo de referência para comparar conjuntos de variáveis. A escolha dos algoritmos finais pertence ao NB_4. Este notebook corre sozinho a partir da raiz do projeto ou da pasta `notebooks/`.

No NB_2 fizemos **transformação** (corrigir, preencher, codificar e escalar). Aqui fazemos **seleção**: escolhemos quais das colunas resultantes entram no modelo. Mantemos as duas etapas separadas para perceber o efeito de cada decisão.

## 1. Reconstruir a base do NB_2

Reutilizamos as decisões já justificadas no NB_2: validar ficheiros; corrigir médias históricas quando existe uma identidade verificável; marcar gastos ou intervalos impossíveis como ausentes; excluir apenas as 65 linhas de treino quase vazias; retirar identificadores, data absoluta, alvo e random_noise das variáveis de base; fazer o corte cronológico com margem de 90 dias.

A opção de juntar atributos de clientes fica desligada. A ficha pode ter sido atualizada depois de algumas compras antigas, e a sua utilidade ainda não foi demonstrada. O NB_4 pode fazer uma comparação separada após confirmar essa disponibilidade temporal.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if not (ROOT / "data" / "Raw" / "train.csv").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "Raw"
assert (DATA / "train.csv").exists(), "Executa a partir da raiz ou de notebooks/."

train_raw = pd.read_csv(DATA / "train.csv")
test_raw = pd.read_csv(DATA / "test.csv")
clientes = pd.read_csv(DATA / "clientes.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
print({name: df.shape for name, df in {
    "train": train_raw, "test": test_raw, "clientes": clientes, "sample": sample
}.items()})

{'train': (6534, 39), 'test': (986, 38), 'clientes': (1143, 13), 'sample': (986, 2)}


In [2]:
TARGET = "repeat_purchase_90d"
assert TARGET in train_raw and TARGET not in test_raw
assert set(train_raw.columns) - set(test_raw.columns) == {TARGET}
assert set(test_raw.columns) - set(train_raw.columns) == set()
assert train_raw["ID"].is_unique and test_raw["ID"].is_unique
assert clientes["customer_id"].is_unique
assert sample["ID"].tolist() == test_raw["ID"].tolist()
assert sample.columns.tolist() == ["ID", TARGET]
assert train_raw[TARGET].isin([0, 1]).all()
print("Estrutura, IDs e ordem da submissão: OK")

Estrutura, IDs e ordem da submissão: OK


In [3]:
def clean_fixed(frame):
    out = frame.copy()
    out["purchase_date"] = pd.to_datetime(out["purchase_date"], errors="coerce")

    col = "prior_mean_purchase_value"
    invalid_mean = out[col].lt(0) | out[col].ge(1e8)
    valid_parts = out["prior_purchase_count"].gt(0) & out["prior_total_spend"].ge(0)
    reconstructed = out["prior_total_spend"] / out["prior_purchase_count"].replace(0, np.nan)
    out.loc[invalid_mean & valid_parts, col] = reconstructed[invalid_mean & valid_parts]
    out.loc[invalid_mean & ~valid_parts, col] = np.nan

    outside_bounds = (
        out[col].lt(out["prior_min_purchase_value"] - 0.05) |
        out[col].gt(out["prior_max_purchase_value"] + 0.05)
    )
    reconstructed_in_bounds = (
        reconstructed.ge(out["prior_min_purchase_value"] - 0.05) &
        reconstructed.le(out["prior_max_purchase_value"] + 0.05)
    )
    repair_bounds = outside_bounds & reconstructed_in_bounds & valid_parts
    out.loc[repair_bounds, col] = reconstructed[repair_bounds]

    identity_disagrees = (out[col] - reconstructed).abs().gt(
        np.maximum(0.05, 0.01 * reconstructed.abs())
    )
    repair_identity = identity_disagrees & reconstructed_in_bounds & valid_parts
    out.loc[repair_identity, col] = reconstructed[repair_identity]

    invalid_spend = (
        out["spend_365d"].lt(0) |
        out["spend_365d"].ge(1e8) |
        out["spend_365d"].gt(out["prior_total_spend"] + 0.05)
    )
    out["spend_365d"] = out["spend_365d"].mask(invalid_spend)

    for gap in ("days_since_previous_purchase", "prior_mean_gap_days"):
        out[gap] = out[gap].mask(out[gap].lt(0))
    return out

train_clean = clean_fixed(train_raw)
test_clean = clean_fixed(test_raw)
for name, before, after in (("train", train_raw, train_clean), ("test", test_raw, test_clean)):
    changed = {c: int((before[c].notna() & after[c].isna()).sum())
               for c in ("spend_365d", "days_since_previous_purchase", "prior_mean_gap_days")}
    repaired = int((before["prior_mean_purchase_value"].notna() &
                    after["prior_mean_purchase_value"].notna() &
                    (before["prior_mean_purchase_value"] -
                     after["prior_mean_purchase_value"]).abs().gt(0.05)).sum())
    print(name, "médias reconstruídas:", repaired, "| valores tornados em falta:", changed)

train médias reconstruídas: 78 | valores tornados em falta: {'spend_365d': 69, 'days_since_previous_purchase': 16, 'prior_mean_gap_days': 16}
test médias reconstruídas: 11 | valores tornados em falta: {'spend_365d': 11, 'days_since_previous_purchase': 2, 'prior_mean_gap_days': 2}


In [4]:
purchase_totals = ["Total a Pagar", "Total Bruto", "Total Liquido", "Total Imp_", "Custo"]
no_purchase_info = train_clean[purchase_totals].isna().all(axis=1)
usable = ~no_purchase_info & train_clean["purchase_date"].notna() & train_clean[TARGET].notna()
print("Linhas excluídas do treino:", int((~usable).sum()))
print("Linhas de treino utilizáveis:", int(usable.sum()))
assert len(test_clean) == len(sample)
assert test_clean["purchase_date"].notna().all()

Linhas excluídas do treino: 65
Linhas de treino utilizáveis: 6469


In [5]:
CLIENT_COLUMNS = ["customer_id", "customer_since_year", "country_or_region"]

def make_features(clean_frame, include_clientes=False):
    out = clean_frame.copy()
    if include_clientes:
        out = out.merge(clientes[CLIENT_COLUMNS], on="customer_id",
                        how="left", validate="m:1", sort=False)
        assert len(out) == len(clean_frame)
    X = out.drop(columns=["ID", "customer_id", "purchase_date", TARGET, "random_noise"],
                 errors="ignore")
    text_cols = X.select_dtypes(include=["str", "object", "category"]).columns
    for col in text_cols:
        X[col] = X[col].astype(object).where(X[col].notna(), np.nan)
    return X

X_all = make_features(train_clean)
X_test = make_features(test_clean)
X = X_all.loc[usable].copy()
y = train_clean.loc[usable, TARGET].astype(int)
dates = train_clean.loc[usable, "purchase_date"]
assert X.columns.tolist() == X_test.columns.tolist()
assert len(X_test) == len(test_raw)
print("Variáveis de base:", X.shape[1], "| numéricas:",
      X.select_dtypes(include="number").shape[1],
      "| categóricas:", X.select_dtypes(exclude="number").shape[1])

Variáveis de base: 34 | numéricas: 28 | categóricas: 6


In [6]:
boundary = dates.sort_values().iloc[int(0.8 * len(dates))]
purge_start = boundary - pd.Timedelta(days=90)
fit_rows = dates < purge_start
validation_rows = dates >= boundary

X_fit, y_fit = X.loc[fit_rows], y.loc[fit_rows]
X_val, y_val = X.loc[validation_rows], y.loc[validation_rows]
assert dates.loc[fit_rows].max() < purge_start
assert dates.loc[validation_rows].min() >= boundary
assert not X_fit.index.intersection(X_val.index).size

print("Início da validação:", boundary.date())
print("Última data do treino:", dates.loc[fit_rows].max().date())
print("Linhas para ajuste:", len(X_fit), "| validação:", len(X_val),
      "| margem:", int((~fit_rows & ~validation_rows).sum()))

Início da validação: 2023-06-17
Última data do treino: 2023-03-18
Linhas para ajuste: 5024 | validação: 1298 | margem: 147


In [7]:
def make_preprocessor(X_reference, scale_numeric=True):
    numeric_cols = X_reference.select_dtypes(include="number").columns.tolist()
    category_cols = X_reference.select_dtypes(exclude="number").columns.tolist()
    numeric_steps = [
        ("impute", SimpleImputer(strategy="median", add_indicator=True,
                                 keep_empty_features=True))
    ]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    numeric_pipe = Pipeline(numeric_steps)
    category_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="__missing__",
                                 keep_empty_features=True)),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=20))
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric_cols),
        ("category", category_pipe, category_cols)
    ])

## 2. Escolher a métrica e as divisões temporais

O alvo vale 1 em cerca de 56% das compras. Usamos **ROC-AUC** para comparar a ordenação das probabilidades sem depender de um limiar fixo. A **balanced accuracy** com limiar 0,5 é uma medida complementar para as duas classes.

Criamos três divisões cronológicas dentro de `X_fit`. Cada treino termina pelo menos 90 dias antes da sua validação, porque o alvo observa a recompra nos 90 dias seguintes. A imputação, a codificação, a escala e a seleção são aprendidas novamente em cada treino. `X_val`, o período mais recente separado no NB_2, fica reservado até escolhermos o método.

Esta validação temporal substitui uma divisão aleatória porque queremos prever compras futuras. Usar o mesmo bloco para selecionar variáveis e para estimar o resultado seria otimista: até uma variável sem utilidade pode parecer boa por acaso. Por isso, cada avaliação inclui a seleção **dentro** do respetivo treino, e o período `X_val` fica fora da comparação de candidatos.

In [8]:
from sklearn.feature_selection import (
    SelectKBest, f_classif, RFE, SelectFromModel, VarianceThreshold
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix

date_fit = dates.loc[X_fit.index]
ordered = date_fit.sort_values()
fractions = (0.50, 0.65, 0.80)
starts = [ordered.iloc[int(q * len(ordered))] for q in fractions]
ends = starts[1:] + [date_fit.max() + pd.Timedelta(days=1)]
inner_folds = []
for fold_number, (start, end) in enumerate(zip(starts, ends), start=1):
    # A margem impede que o alvo das últimas compras de treino alcance a validação.
    train_rows = date_fit < start - pd.Timedelta(days=90)
    check_rows = (date_fit >= start) & (date_fit < end)
    train_index = date_fit.index[train_rows]
    check_index = date_fit.index[check_rows]
    assert len(train_index) > 0 and len(check_index) > 0
    assert date_fit.loc[train_index].max() < start - pd.Timedelta(days=90)
    assert not train_index.intersection(check_index).size
    assert y_fit.loc[train_index].nunique() == y_fit.loc[check_index].nunique() == 2
    inner_folds.append((train_index, check_index))
    print(f"Divisão {fold_number}: treino {len(train_index)}, avaliação {len(check_index)}, "
          f"avaliação de {start.date()} a {(end - pd.Timedelta(days=1)).date()}")

Divisão 1: treino 2347, avaliação 760, avaliação de 2019-08-24 a 2020-10-15
Divisão 2: treino 3132, avaliação 755, avaliação de 2020-10-16 a 2021-10-14
Divisão 3: treino 3831, avaliação 1005, avaliação de 2021-10-15 a 2023-03-18


## 3. Métodos de seleção comparados

**Sem seleção** é a referência: mantemos todas as colunas transformadas, exceto as constantes. `VarianceThreshold` remove essas colunas porque não distinguem as compras neste treino; é aplicado novamente em cada divisão.

**Filter — SelectKBest com teste F.** Mede a associação de cada coluna com o alvo **isoladamente**, sem usar o modelo que virá depois. Como o alvo é categórico (0/1), o teste F/ANOVA é uma opção dos slides para entradas numéricas. Aplicamo-lo depois do one-hot: cada categoria passa a ser uma coluna numérica 0/1. Os slides também indicam qui-quadrado ou informação mútua para categorias originais; aqui o teste F é uma escolha simples e uniforme para as colunas já transformadas. As pontuações servem para ordenar colunas, não para provar causalidade ou significância estatística. Comparamos 20 e 40 colunas.

O filtro é rápido, mas uma coluna com pontuação baixa pode ser útil em conjunto com outra; e duas colunas muito correlacionadas podem repetir informação. É por isso que não eliminamos automaticamente variáveis olhando apenas para correlações do NB_1.

**Wrapper — RFE.** Ajusta uma regressão logística, ordena as colunas pelos coeficientes e vai retirando as menos importantes até restarem 20 ou 40. Considera o modelo, mas exige vários ajustes. Se fosse aplicado uma vez antes de dividir os dados, as escolhas poderiam aproveitar coincidências do conjunto avaliado. Aqui o RFE é reajustado dentro de cada treino temporal.

**Embedded — L1.** A penalização faz parte do ajuste de uma regressão logística e pode levar coeficientes exatamente a zero; `SelectFromModel` conserva os restantes. Um `C` menor significa penalização mais forte e, em geral, menos colunas. Comparamos `C=0.1` e `C=0.5`. A regressão Ridge/L2 estudada nos slides tende a reduzir coeficientes sem os anular, por isso não a usamos como seletor direto.

Os valores 20, 40 e `C` são **candidatos de comparação**, não exigências do enunciado. A imputação, a codificação, a escala, a remoção de constantes e a seleção ficam na mesma `Pipeline` e são aprendidas só com o treino de cada divisão.

In [9]:
def reference_model():
    return LogisticRegression(solver="liblinear", max_iter=2000, random_state=42)

SELECTORS = {
    "Todas": lambda: "passthrough",
    "Filter F · 20": lambda: SelectKBest(score_func=f_classif, k=20),
    "Filter F · 40": lambda: SelectKBest(score_func=f_classif, k=40),
    "Wrapper RFE · 20": lambda: RFE(reference_model(), n_features_to_select=20, step=0.25),
    "Wrapper RFE · 40": lambda: RFE(reference_model(), n_features_to_select=40, step=0.25),
    "Embedded L1 · C=0.1": lambda: SelectFromModel(
        LogisticRegression(solver="liblinear", l1_ratio=1, C=0.1,
                           max_iter=2000, random_state=42), threshold=1e-8
    ),
    "Embedded L1 · C=0.5": lambda: SelectFromModel(
        LogisticRegression(solver="liblinear", l1_ratio=1, C=0.5,
                           max_iter=2000, random_state=42), threshold=1e-8
    ),
}

def build_pipeline(name, X_reference):
    # Todas as etapas aprendidas ficam dentro do treino de cada divisão.
    return Pipeline([
        ("preprocess", make_preprocessor(X_reference, scale_numeric=True)),
        ("variance", VarianceThreshold()),
        ("select", SELECTORS[name]()),
        ("model", reference_model()),
    ])

def selected_names(fitted_pipeline):
    # Recupera os nomes depois de retirar constantes e aplicar o seletor.
    names = fitted_pipeline.named_steps["preprocess"].get_feature_names_out()
    names = names[fitted_pipeline.named_steps["variance"].get_support()]
    selector = fitted_pipeline.named_steps["select"]
    if selector != "passthrough":
        names = names[selector.get_support()]
    return list(names)

## 4. Comparar os métodos sem tocar no período reservado

Cada candidato passa pelas **mesmas três divisões**. Em cada divisão, a preparação e a seleção são aprendidas de novo com as linhas antigas; as probabilidades são avaliadas apenas no bloco seguinte. Guardamos ROC-AUC, balanced accuracy e número de colunas selecionadas. Não escolhemos um método com base em resultados de X_val.

Usamos as mesmas divisões para todos os métodos: assim, diferenças de resultado não vêm de terem sido avaliados em períodos diferentes. O RFE e o L1 usam regressão logística como referência nesta etapa; um modelo diferente no NB_4 poderá preferir outro conjunto.

In [10]:
records = []
for name in SELECTORS:
    for fold_number, (train_index, check_index) in enumerate(inner_folds, start=1):
        # O seletor é ajustado de novo; nunca vê o bloco usado para avaliar.
        pipeline = build_pipeline(name, X_fit.loc[train_index])
        pipeline.fit(X_fit.loc[train_index], y_fit.loc[train_index])
        probabilities = pipeline.predict_proba(X_fit.loc[check_index])[:, 1]
        predicted = (probabilities >= 0.5).astype(int)
        records.append({
            "método": name,
            "divisão": fold_number,
            "ROC-AUC": roc_auc_score(y_fit.loc[check_index], probabilities),
            "balanced accuracy": balanced_accuracy_score(y_fit.loc[check_index], predicted),
            "colunas": len(selected_names(pipeline)),
        })
        print(f"{name:22s} | divisão {fold_number} | AUC {records[-1]['ROC-AUC']:.4f}"
              f" | colunas {records[-1]['colunas']}")

fold_results = pd.DataFrame(records)

Todas                  | divisão 1 | AUC 0.7658 | colunas 76
Todas                  | divisão 2 | AUC 0.7644 | colunas 79
Todas                  | divisão 3 | AUC 0.7925 | colunas 84


Filter F · 20          | divisão 1 | AUC 0.7532 | colunas 20


Filter F · 20          | divisão 2 | AUC 0.7578 | colunas 20


Filter F · 20          | divisão 3 | AUC 0.8001 | colunas 20
Filter F · 40          | divisão 1 | AUC 0.7655 | colunas 40


Filter F · 40          | divisão 2 | AUC 0.7697 | colunas 40
Filter F · 40          | divisão 3 | AUC 0.7936 | colunas 40
Wrapper RFE · 20       | divisão 1 | AUC 0.7621 | colunas 20


...

Wrapper RFE · 40       | divisão 3 | AUC 0.7968 | colunas 40


Embedded L1 · C=0.1    | divisão 1 | AUC 0.7580 | colunas 23
Embedded L1 · C=0.1    | divisão 2 | AUC 0.7777 | colunas 29


Embedded L1 · C=0.1    | divisão 3 | AUC 0.7892 | colunas 30


Embedded L1 · C=0.5    | divisão 1 | AUC 0.7674 | colunas 50
Embedded L1 · C=0.5    | divisão 2 | AUC 0.7781 | colunas 53


Embedded L1 · C=0.5    | divisão 3 | AUC 0.7898 | colunas 64


In [11]:
comparison = (fold_results.groupby("método", sort=False)
              .agg(media_auc=("ROC-AUC", "mean"),
                   desvio_auc=("ROC-AUC", "std"),
                   media_balanced_accuracy=("balanced accuracy", "mean"),
                   media_colunas=("colunas", "mean"),
                   minimo_auc=("ROC-AUC", "min"),
                   maximo_auc=("ROC-AUC", "max")))
comparison["erro_padrao_auc"] = comparison["desvio_auc"] / np.sqrt(len(inner_folds))
comparison = comparison.sort_values("media_auc", ascending=False)
print(comparison.round(4).to_string())

                     media_auc  desvio_auc  media_balanced_accuracy  media_colunas  minimo_auc  maximo_auc  erro_padrao_auc
método                                                                                                                     
Embedded L1 · C=0.5     0.7784      0.0112                   0.7125        55.6667      0.7674      0.7898           0.0065
Filter F · 40           0.7763      0.0152                   0.7142        40.0000      0.7655      0.7936           0.0088
Wrapper RFE · 40        0.7761      0.0182                   0.7011        40.0000      0.7629      0.7968           0.0105
Embedded L1 · C=0.1     0.7750      0.0158                   0.7169        27.3333      0.7580      0.7892           0.0091
Todas                   0.7742      0.0158                   0.7045        79.6667      0.7644      0.7925           0.0091
Filter F · 20           0.7704      0.0259                   0.7088        20.0000      0.7532      0.8001           0.0149
Wrapper 

## 5. Escolher o método

A maior ROC-AUC média foi **0,7784** com L1 (`C=0.5`). As diferenças entre os melhores métodos são pequenas; a referência com todas as colunas obteve 0,7742. Os candidatos com apenas 20 colunas tiveram resultados inferiores.

Para equilibrar desempenho e simplicidade, aceitamos médias até **um erro padrão** abaixo da melhor e escolhemos, nesse grupo, o método com menos colunas. O erro padrão mostrado na tabela é o desvio das três ROC-AUC dividido por √3. Esta é uma regra prática definida sobre os resultados internos, não uma garantia estatística: os períodos são poucos e os treinos sobrepõem-se.

In [12]:
# A escolha usa apenas os resultados das divisões internas; X_val continua reservado.
best_name = comparison.index[0]
cutoff = comparison.loc[best_name, "media_auc"] - comparison.loc[best_name, "erro_padrao_auc"]
eligible = comparison.loc[comparison["media_auc"] >= cutoff].copy()
chosen_name = (eligible.sort_values(["media_colunas", "media_auc"],
                                    ascending=[True, False]).index[0])
print("Maior AUC média:", best_name,
      f"({comparison.loc[best_name, 'media_auc']:.4f})")
print("Limite de aceitação:", round(cutoff, 4))
print("Métodos dentro do limite:", list(eligible.index))
print("Método escolhido:", chosen_name)

Maior AUC média: Embedded L1 · C=0.5 (0.7784)
Limite de aceitação: 0.772
Métodos dentro do limite: ['Embedded L1 · C=0.5', 'Filter F · 40', 'Wrapper RFE · 40', 'Embedded L1 · C=0.1', 'Todas']
Método escolhido: Embedded L1 · C=0.1


**Decisão:** L1 com `C=0.1`. Nas divisões internas, obteve ROC-AUC médio de **0,7750** com **27,3 colunas** em média; ficou dentro da margem definida e foi o candidato aceite com menos colunas. O L1 com `C=0.5` teve média ligeiramente maior, mas usou cerca de 55,7 colunas.

## 6. Verificar no período reservado

Ajustamos o método escolhido em todo `X_fit` e avaliamos uma única vez em `X_val`. Fazemos a mesma avaliação com todas as colunas para ter uma referência. Esta verificação ajuda a detetar uma escolha que só funcionou nas divisões internas; não usamos o resultado para trocar novamente de método. A balanced accuracy usa limiar 0,5 apenas como diagnóstico; o NB_4 poderá estudar outro limiar.

In [13]:
chosen_pipeline = build_pipeline(chosen_name, X_fit)
chosen_pipeline.fit(X_fit, y_fit)
reference_pipeline = build_pipeline("Todas", X_fit)
reference_pipeline.fit(X_fit, y_fit)

holdout_rows = []
for label, pipeline in (("Seleção escolhida", chosen_pipeline),
                        ("Todas as colunas", reference_pipeline)):
    probabilities = pipeline.predict_proba(X_val)[:, 1]
    predicted = (probabilities >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, predicted, labels=[0, 1]).ravel()
    holdout_rows.append({
        "conjunto": label,
        "ROC-AUC": roc_auc_score(y_val, probabilities),
        "balanced accuracy": balanced_accuracy_score(y_val, predicted),
        "colunas": len(selected_names(pipeline)),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
    })
holdout = pd.DataFrame(holdout_rows).set_index("conjunto")
print(holdout.round(4).to_string())

                   ROC-AUC  balanced accuracy  colunas   TP   FP   FN   TN
conjunto                                                                  
Seleção escolhida   0.7901             0.7244       38  543  126  240  389
Todas as colunas    0.7863             0.7172       92  556  142  227  373


**Leitura:** na validação reservada, a seleção L1 obteve ROC-AUC **0,7901** com **38 colunas transformadas**. Com todas as 92 colunas, o ROC-AUC foi **0,7863**. A diferença é pequena e não prova melhoria futura; a redução do número de colunas é o resultado mais claro. O seletor escolheu 38 colunas neste ajuste, em vez da média interna de 27,3, porque foi reajustado com mais dados.

## 7. Variáveis selecionadas

A seleção atua sobre **colunas transformadas**: uma variável categórica pode gerar várias colunas one-hot, e uma numérica com nulos pode originar um indicador de ausência. A tabela liga cada coluna escolhida à variável original. Selecionar algumas categorias de uma variável não significa selecionar a variável inteira; deixar uma categoria de fora também não prova que nunca será útil.

Esta lista descreve o ajuste atual em `X_fit`. No NB_4, devemos reutilizar o **método e os seus parâmetros** dentro de cada treino, em vez de fixar esta lista de nomes.

In [14]:
final_names = selected_names(chosen_pipeline)
category_cols = X_fit.select_dtypes(exclude="number").columns.tolist()

def original_column(encoded_name):
    # Os prefixos são os nomes atribuídos pelo ColumnTransformer.
    prefix = "numeric__missingindicator_"
    if encoded_name.startswith(prefix):
        return encoded_name[len(prefix):]
    prefix = "numeric__"
    if encoded_name.startswith(prefix):
        return encoded_name[len(prefix):]
    prefix = "category__"
    if encoded_name.startswith(prefix):
        remainder = encoded_name[len(prefix):]
        for col in sorted(category_cols, key=len, reverse=True):
            if remainder.startswith(col + "_"):
                return col
    return "(verificar nome)"

selection_table = pd.DataFrame({
    "coluna transformada": final_names,
    "variável original": [original_column(name) for name in final_names],
})
assert not selection_table["variável original"].eq("(verificar nome)").any()
print(selection_table.to_string(index=False))
print("\nVariáveis originais representadas:",
      selection_table["variável original"].nunique())
print(selection_table["variável original"].value_counts().to_string())

                                  coluna transformada              variável original
                               numeric__Total Liquido                  Total Liquido
                                  numeric__Total Imp_                     Total Imp_
                                numeric__Total Portes                   Total Portes
                           numeric__Total Desc_ Linha              Total Desc_ Linha
                                 numeric__N_ Detalhes                    N_ Detalhes
              numeric__invoice_count_current_purchase invoice_count_current_purchase
                        numeric__customer_tenure_days           customer_tenure_days
                        numeric__prior_purchase_count           prior_purchase_count
                   numeric__prior_mean_purchase_value      prior_mean_purchase_value
                    numeric__prior_std_purchase_value       prior_std_purchase_value
                    numeric__prior_min_purchase_value       prior

## 8. Conclusão e ligação ao NB_4

O método candidato é **L1 com `C=0.1`**: selecionou 38 das 92 colunas transformadas e teve ROC-AUC de 0,7901 na validação temporal, contra 0,7863 com todas as colunas. Estas 38 colunas representam 25 variáveis originais. A diferença de desempenho é pequena, por isso a seleção ainda deve ser comparada com a versão sem seleção no NB_4.

A principal regra a conservar é colocar **pré-processamento e seleção dentro de cada treino**, respeitando a margem temporal de 90 dias. O NB_4 compara os modelos com a mesma estratégia temporal; o NB_5 ajusta o processo escolhido para prever o teste.